# 02 Preprocessing

Executable reporting notebook for preprocessing, keyword relabeling, aspect taxonomy derivation, weak aspect labeling, and refinement quality control. It reads generated outputs and figures without training SVM or IndoBERT.


## CRISP-DM Stage

Data Preparation. This notebook verifies generated preprocessing outputs and documents limitations before model training.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, Markdown, display


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "CLAUDE.md").exists() and (candidate / "ml-service").exists():
            return candidate
    raise RuntimeError("Could not find SentiRank project root from current working directory.")


PROJECT_ROOT = find_project_root()
ML_SERVICE_DIR = PROJECT_ROOT / "ml-service"
DATASETS_DIR = PROJECT_ROOT / "datasets"
DOCS_FIGURES_DIR = PROJECT_ROOT / "docs" / "figures"


def load_json(path: Path):
    if not path.exists():
        display(Markdown(f"Missing JSON: `{path.relative_to(PROJECT_ROOT)}`"))
        return None
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


def load_csv(path: Path, **kwargs):
    if not path.exists():
        display(Markdown(f"Missing CSV: `{path.relative_to(PROJECT_ROOT)}`"))
        return None
    return pd.read_csv(path, **kwargs)


def show_json_as_table(payload, title: str):
    if payload is None:
        return
    display(Markdown(f"### {title}"))
    try:
        table = pd.json_normalize(payload, sep=".").T.reset_index()
        table.columns = ["field", "value"]
        display(table)
    except Exception as exc:
        display(Markdown(f"Could not tabulate JSON payload: `{exc}`"))
        display(payload)


def show_csv_preview(path: Path, title: str, rows: int = 10):
    data = load_csv(path)
    if data is None:
        return None
    display(Markdown(f"### {title}"))
    display(data.head(rows))
    if len(data) > rows:
        display(Markdown(f"Rows: `{len(data)}`. Showing first `{rows}` rows."))
    return data


def show_figures(figure_paths: list[Path], title: str):
    display(Markdown(f"## {title}"))
    shown = 0
    for figure_path in figure_paths:
        if figure_path.exists():
            display(Markdown(f"**{figure_path.relative_to(PROJECT_ROOT)}**"))
            display(Image(filename=str(figure_path)))
            shown += 1
        else:
            display(Markdown(f"Missing figure: `{figure_path.relative_to(PROJECT_ROOT)}`"))
    if shown == 0:
        display(Markdown("No figures were available to display."))

print(f"Project root: {PROJECT_ROOT}")


## Reproducible Pipeline Commands

The commands below are the reproducible preprocessing entrypoints. They are displayed by default. Setting `RUN_PREPROCESSING_PIPELINE = True` will execute them and regenerate ignored processed datasets, so keep it disabled unless intentionally rerunning preprocessing.


In [ ]:
import subprocess
import sys

PREPROCESSING_COMMANDS = [
    [sys.executable, "scripts/relabel_by_keywords.py", "--input", "../datasets/raw/reviews_raw_labeled.csv", "--output", "../datasets/processed/reviews_relabelled.csv"],
    [sys.executable, "scripts/preprocess_indobert.py", "--input", "../datasets/processed/reviews_relabelled.csv", "--output", "../datasets/processed/reviews_preprocessed_indobert.csv"],
    [sys.executable, "scripts/preprocess_svm.py", "--input", "../datasets/processed/reviews_preprocessed_indobert.csv", "--output", "../datasets/processed/reviews_preprocessed_svm.csv"],
    [sys.executable, "scripts/derive_aspect_taxonomy.py", "--input", "../datasets/processed/reviews_final.csv", "--output-summary", "../datasets/outputs/eda/02_preprocessing/aspect_taxonomy_derivation_summary.json", "--output-keywords", "../datasets/outputs/eda/02_preprocessing/aspect_taxonomy_candidate_terms.csv"],
    [sys.executable, "scripts/label_aspects_by_keywords.py", "--input", "../datasets/processed/reviews_final.csv", "--output", "../datasets/processed/reviews_with_aspect_labels_refined.csv", "--summary-output", "../datasets/outputs/eda/02_preprocessing/aspect_labeling_refined_summary.json"],
]

for command in PREPROCESSING_COMMANDS:
    display(Markdown("```bash\n" + " ".join(command) + "\n```"))

RUN_PREPROCESSING_PIPELINE = False
if RUN_PREPROCESSING_PIPELINE:
    for command in PREPROCESSING_COMMANDS:
        result = subprocess.run(command, cwd=ML_SERVICE_DIR, text=True, capture_output=True, check=False)
        print("$ " + " ".join(command))
        print(result.stdout)
        if result.stderr:
            print(result.stderr)
        if result.returncode != 0:
            raise RuntimeError(f"Command failed with return code {result.returncode}")
else:
    display(Markdown("Preprocessing commands were not executed. This notebook is in reporting mode."))


## Load Preprocessing Summaries

The notebook reads relabeling, preprocessing, taxonomy, and aspect-label quality summaries from `datasets/outputs/eda/02_preprocessing/`.


In [ ]:
EDA02_DIR = DATASETS_DIR / "outputs" / "eda" / "02_preprocessing"
summary_paths = {
    "Relabeling summary": EDA02_DIR / "relabeling_summary.json",
    "Preprocessing summary": EDA02_DIR / "preprocessing_summary.json",
    "Aspect taxonomy derivation summary": EDA02_DIR / "aspect_taxonomy_derivation_summary.json",
    "Aspect labeling summary": EDA02_DIR / "aspect_labeling_summary.json",
    "Refined aspect labeling summary": EDA02_DIR / "aspect_labeling_refined_summary.json",
    "General fallback analysis": EDA02_DIR / "general_fallback_analysis.json",
}

preprocessing_summaries = {name: load_json(path) for name, path in summary_paths.items()}
for name, payload in preprocessing_summaries.items():
    show_json_as_table(payload, name)


## Metric Tables

These aggregate CSV/JSON files are suitable for thesis tables and future dashboard visualization.


In [ ]:
metric_files = [
    "label_distribution_before_relabeling.csv",
    "label_distribution_after_relabeling.csv",
    "text_length_before_after_cleaning.csv",
    "aspect_label_distribution.csv",
    "aspect_label_distribution_refined.csv",
    "aspect_by_sentiment_distribution.csv",
    "aspect_by_sentiment_distribution_refined.csv",
    "aspect_label_confidence_distribution.csv",
    "aspect_taxonomy_candidate_terms.csv",
    "general_fallback_terms.csv",
]

loaded_eda02_tables = {}
for filename in metric_files:
    table = show_csv_preview(EDA02_DIR / filename, filename, rows=15)
    if table is not None:
        loaded_eda02_tables[filename] = table

for filename in [
    "label_distribution_before_relabeling.json",
    "label_distribution_after_relabeling.json",
    "text_length_before_after_cleaning.json",
    "aspect_label_distribution_refined.json",
    "aspect_label_confidence_distribution.json",
]:
    show_json_as_table(load_json(EDA02_DIR / filename), filename)


## Preprocessing and Aspect Figures

Figures are loaded from `docs/figures/02_preprocessing/`.


In [ ]:
FIG02_DIR = DOCS_FIGURES_DIR / "02_preprocessing"
figure_files = [
    "label_distribution_before_relabeling.png",
    "label_distribution_after_relabeling.png",
    "relabeling_change_summary.png",
    "text_length_before_after_cleaning.png",
    "aspect_candidate_terms.png",
    "aspect_label_distribution.png",
    "aspect_by_sentiment_distribution.png",
    "aspect_label_distribution_refined.png",
    "aspect_by_sentiment_distribution_refined.png",
    "aspect_label_confidence_distribution.png",
    "general_fallback_top_terms.png",
    "general_rate_before_after.png",
]
show_figures([FIG02_DIR / filename for filename in figure_files], "Preprocessing Figures")


## Interpretation Notes

This cell turns the generated summaries into thesis-facing notes.


In [ ]:
notes = []

relabel = preprocessing_summaries.get("Relabeling summary") or {}
if relabel:
    changed_count = relabel.get("changed_label_count")
    changed_pct = relabel.get("changed_label_percentage")
    notes.append(f"Keyword relabeling changed `{changed_count}` rows (`{changed_pct:.4f}%`) while preserving `initial_sentiment`.")
    before = relabel.get("label_distribution_before")
    after = relabel.get("label_distribution_after")
    if before and after:
        notes.append(f"Label distribution before relabeling: `{before}`; after relabeling: `{after}`.")

preprocess = preprocessing_summaries.get("Preprocessing summary") or {}
if preprocess:
    notes.append(f"Preprocessing summary fields available: `{list(preprocess.keys())}`.")

aspect_summary = preprocessing_summaries.get("Aspect labeling summary") or {}
refined_summary = preprocessing_summaries.get("Refined aspect labeling summary") or {}
if aspect_summary:
    notes.append(f"Initial weak aspect General fallback rate: `{aspect_summary.get('general_label_percentage')}`%.")
if refined_summary:
    notes.append(f"Refined weak aspect General fallback rate: `{refined_summary.get('general_label_percentage')}`%.")
    notes.append(f"Rows with keyword match after refinement: `{refined_summary.get('rows_with_keyword_match')}`.")

confidence = loaded_eda02_tables.get("aspect_label_confidence_distribution.csv")
if confidence is not None:
    notes.append("Aspect confidence distribution is available; later SVM training should prioritize high/medium confidence labels or a balanced refined subset.")

notes.append("The aspect taxonomy is exploratory and weak-labeled. It is not expert-validated ground truth and must not be used directly as final AHP/Fuzzy AHP criteria.")
notes.append("`General` is a fallback label only, not a decision criterion for AHP/Fuzzy AHP.")
notes.append("Next steps: use `03_indobert_sentiment_modeling.ipynb` for final sentiment model selection and `04_svm_aspect_classification.ipynb` later for aspect classifier training.")

display(Markdown("\n".join(f"- {note}" for note in notes)))
